### This is the cleaning and preparation of summary statistics

In [93]:
import pandas as pd

Parameters include
1. Grouping results by 
State
Year
Month

2. All of the US

3. Demographics
Filter to: 
Single-Year Ages < 1 for all infants
Infant Age Groups <1 day to 7-27 days
Single Race 6

4. All years and months

5. All weekday, autopsy, place of death

6. All underlying cause of death

7. All MCD

8. Export to CSV
Don't show totals
Show Zero Values
Show Suppressed Values

In [94]:
#Import Raw Data - Already Implemented
raw = pd.read_csv(r"Data\Raw\infant_raw_state.csv")

raw_neonatal = pd.read_csv(r"Data\Raw\neonatal_raw_state.csv")

raw_births = pd.read_csv(r"Data\Raw\births_raw.csv")


In [95]:
print(raw.columns)
print(raw_births.columns)

Index(['Notes', 'State', 'State Code', 'Year', 'Year Code', 'Month',
       'Month Code', 'Deaths', 'Population', 'Crude Rate',
       'Crude Rate Lower 95% Confidence Interval',
       'Crude Rate Upper 95% Confidence Interval'],
      dtype='object')
Index(['Notes', 'State of Residence', 'State of Residence Code', 'Year',
       'Year Code', 'Month', 'Month Code', 'Births'],
      dtype='object')


In [96]:
renaming_dict = {'State':'state',
                 'State of Residence':'state',
                 'State Code':'state_code',
                 'State of Residence Code':'state_code',
                 'Year':'year',
                 'Month':'month',
                 'Births':'births',
                'Deaths':'deaths',
                'Month Code':'month_code'
                 }

def clean_month(month_str):
    if pd.isnull(month_str):
        return 0
    
    if 'Jan' in month_str:
        return 1
    elif 'Feb' in month_str:
        return 2
    elif 'Mar' in month_str:
        return 3
    elif 'Apr' in month_str:
        return 4
    elif 'May' in month_str:
        return 5
    elif 'Jun' in month_str:
        return 6
    elif 'Jul' in month_str:
        return 7
    elif 'Aug' in month_str:
        return 8
    elif 'Sep' in month_str:
        return 9
    elif 'Oct' in month_str:
        return 10
    elif 'Nov' in month_str:
        return 11
    elif 'Dec' in month_str:
        return 12
    else:
        return 0

#Cleaning Data
def clean_infant_data(raw_df):
    # Clean the raw data
    df = raw_df.copy(deep = True)
    df['Deaths'] = df['Deaths'].replace('Suppressed', 0).fillna(0).astype(int)
    df = df.rename(columns=renaming_dict)
    df['month'] = df['month'].apply(clean_month)

    df['state_code'] = df['state_code'].astype(str).str.replace('.0','',regex=False).str.zfill(2)
    return df

def clean_births_data(raw_df):
    # Clean the raw data
    df = raw_df.copy(deep = True)
    df = df.rename(columns=renaming_dict)

    df = df.drop(columns = ['month'])
    df['month'] = df['month_code']

    df['state_code'] = df['state_code'].astype(str).str.replace('.0','',regex = False).str.zfill(2)
    return df

In [98]:
#Clean
df = clean_infant_data(raw)

df_neonatal = clean_infant_data(raw_neonatal)

df_births = clean_births_data(raw_births)

#Limit to deaths > 10
#df = df[df['deaths'] > 10]

df_neonatal = df_neonatal[df_neonatal['deaths'] > 10]
print(df.state_code.unique())
print(df_births.state_code.unique())



['01' '02' '04' '05' '06' '08' '09' '10' '11' '12' '13' '15' '16' '17'
 '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30' '31'
 '32' '33' '34' '35' '36' '37' '38' '39' '40' '41' '42' '44' '45' '46'
 '47' '48' '49' '50' '51' '53' '54' '55' '56' 'nan']
['01' '02' '04' '05' '06' '08' '09' '10' '11' '12' '13' '15' '16' '17'
 '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30' '31'
 '32' '33' '34' '35' '36' '37' '38' '39' '40' '41' '42' '44' '45' '46'
 '47' '48' '49' '50' '51' '53' '54' '55' '56' 'nan']


In [99]:
print(df[df['state']=='Texas'].groupby('year')['deaths'].sum())

year
2018.0    2083
2019.0    2075
2020.0    1968
2021.0    1985
2022.0    2243
2023.0    2259
2024.0    2239
Name: deaths, dtype: int32


In [100]:
#Merge with births
df = df.merge(df_births[['year','month','state_code','births']], on=['year', 'month','state_code'], how='left')
df_neonatal = df_neonatal.merge(df_births[['year','month','state_code','births']], on=['year', 'month','state_code'], how='left')

#Calculate Rate
df['rate']  = (df['deaths'] / df['births']) * 1000
df_neonatal['rate']  = (df_neonatal['deaths'] / df_neonatal['births']) * 1000

df = df[['year','month','state_code','state','births','deaths','rate']]
df_neonatal = df_neonatal[['year','month','state_code','state','births','deaths','rate']]

In [101]:
print(df.head())

     year  month state_code    state  births  deaths      rate
0  2018.0      1         01  Alabama  4902.0      40  8.159935
1  2018.0      2         01  Alabama  4342.0      41  9.442653
2  2018.0      3         01  Alabama  4761.0      31  6.511237
3  2018.0      4         01  Alabama  4393.0      32  7.284316
4  2018.0      5         01  Alabama  4785.0      32  6.687565


In [106]:
#Reproducing Table 1
def get_table_1(df,tx = True):
    # Group by Year and State, summing Deaths
    a = df.groupby(['year','state'])[['deaths','births']].sum().reset_index()

    
    # Pivot for Texas
    if tx == True:
        out = a[a['state'] == 'Texas'].groupby('year')[['deaths','births']].sum().reset_index()
    else: 
        out = a[a['state'] != 'Texas'].groupby('year')[['deaths','births']].sum().reset_index()
    out['rate'] = out['deaths'] / out['births'] * 1000
    out['change_yoy_deaths'] = out['deaths'].diff() / out['deaths'].shift(1) * 100
    out['change_yoy_rate'] = out['rate'].diff() / out['rate'].shift(1) * 100

    return out


table_1a,table_1b = get_table_1(df,True),get_table_1(df,False)
#print(table_1a)
#print(table_1b)


table_1c,table_1d = get_table_1(df_neonatal,True),get_table_1(df_neonatal,False)
print(table_1c)
print(table_1d)


     year  deaths    births      rate  change_yoy_deaths  change_yoy_rate
0  2018.0    1382  378624.0  3.650059                NaN              NaN
1  2019.0    1376  377599.0  3.644077          -0.434153        -0.163880
2  2020.0    1300  368190.0  3.530786          -5.523256        -3.108927
3  2021.0    1295  373594.0  3.466330          -0.384615        -1.825542
4  2022.0    1432  389741.0  3.674235          10.579151         5.997848
5  2023.0    1516  387945.0  3.907770           5.865922         6.356030
6  2024.0    1528  390828.0  3.909648           0.791557         0.048053
     year  deaths     births      rate  change_yoy_deaths  change_yoy_rate
0  2018.0   11864  3069747.0  3.864814                NaN              NaN
1  2019.0   11414  3024318.0  3.774074          -3.792987        -2.347839
2  2020.0   10516  2886123.0  3.643642          -7.867531        -3.455991
3  2021.0   10404  2916209.0  3.567646          -1.065044        -2.085738
4  2022.0   10601  2910969.0  3.6